# 03 — Multi-CV Evaluation (Plotly)

From your two Kaggle resume datasets + Ahmed: 8 CVs (`data/sample/cv_*.txt`). Macro-averaged via `scripts/eval_multi_cv.py` → `artifacts/metrics_multi.json`.
Shows *why single-CV 1.00 was misleading* and how diverse CVs surface true hybrid value.

In [ ]:
import json, pathlib, pandas as pd
import plotly.express as px
import plotly.graph_objects as go

p = pathlib.Path('artifacts/metrics_multi.json')
if not p.exists(): p = pathlib.Path('../artifacts/metrics_multi.json')
m = json.loads(p.read_text())
macro = pd.DataFrame(m['macro_avg']['methods']).T.reset_index().rename(columns={'index':'method'})
macro


In [ ]:
# Per-CV nDCG heatmap — which CV is hard?
per = m['per_cv']
rows=[]
for cv, v in per.items():
    for meth, scores in v['methods'].items():
        rows.append({'cv':cv, 'method':meth, 'ndcg@10':scores['ndcg@10'], 'relevant':v['relevant_ge1']})
df = pd.DataFrame(rows)
fig = px.density_heatmap(df, x='method', y='cv', z='ndcg@10', color_continuous_scale='Viridis', title='Per-CV nDCG@10 — HR vs IT vs Ahmed')
fig.update_layout(height=500)
fig.show()
fig.write_html('../docs/images/multi_heatmap.html')

In [ ]:
# Macro vs Single (Ahmed) — bar
single = json.loads(pathlib.Path('artifacts/metrics.json').read_text() if pathlib.Path('artifacts/metrics.json').exists() else pathlib.Path('../artifacts/metrics.json').read_text())
single_df = pd.DataFrame(single['methods']).T.reset_index().rename(columns={'index':'method'})
single_df['scope']='Ahmed only (274/500)'
macro['scope']='Macro 8 CVs (avg 125/500)'
comb = pd.concat([single_df[['method','ndcg@10','scope']], macro[['method','ndcg@10','scope']]])
fig = px.bar(comb, x='method', y='ndcg@10', color='scope', barmode='group', title='Single-CV (Ahmed) vs Macro-averaged (8 CVs) — nDCG@10')
fig.update_layout(height=400, yaxis_range=[0,0.4])
fig.show()
fig.write_html('../docs/images/single_vs_macro.html')

In [ ]:
# Relevant count per CV — bar
rel = pd.DataFrame([{'cv':k, 'relevant':v} for k,v in m['per_cv_relevant'].items()]) if 'per_cv_relevant' in m else pd.DataFrame([{'cv':k, 'relevant':v['relevant_ge1']} for k,v in m['per_cv'].items()])
# fallback
try:
    rel = pd.DataFrame([{'cv':k, 'relevant':v } for k,v in m['per_cv_relevant'].items()])
except:
    rel = pd.DataFrame([{'cv':k, 'relevant': v['relevant_ge1']} for k,v in per.items()])
fig = px.bar(rel, x='cv', y='relevant', color='relevant', color_continuous_scale='Blues', title='Relevant jobs per CV (500 jobs) — Ahmed 274 vs HR 89')
fig.update_layout(height=400, xaxis_tickangle=-20)
fig.show()
fig.write_html('../docs/images/relevant_per_cv.html')